In [3]:
# Load the data
from ingest import load_faq_data
documents = load_faq_data()

In [ ]:
# Convert an array of object into an array of string
texts = [doc["question"]+" "+doc["answer"] for doc in documents]


In [ ]:
# Download the model for embedding
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
from tqdm.auto import tqdm

In [11]:
# Convert the text into embedding in batch of 50
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batch = texts[i:i+batch_size]
  batch_vectors = model.encode(batch)
  vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [12]:
import numpy as np
X = np.array(vectors)

In [13]:
# Creating the index
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [ ]:
# Indexing the data
vs_index.fit(X, documents)

In [15]:
# Searching
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [18]:
# Filtering by course
results = vs_index.search(
  query_vector,
  filter_dict={"course": "llm-zoomcamp"},                      
  num_results=5
  )

In [19]:
# Closing the connection
vs_index.close()